In [ ]:
import pybamm
import numpy as np
import pickle

# pybamm.set_logging_level("INFO")
pybamm.set_logging_level("WARNING")

data_DIR = "../data/"

In [ ]:
# model
model = pybamm.lithium_ion.SPM(
    # {
    #     "SEI": "ec reaction limited",
    # }
)
parameter_values = pybamm.ParameterValues("Mohtat2020")

def nmc_volume_change_mohtat(sto,c_s_max):
    t_change = -1.10/100*(1-sto)
    return t_change

def graphite_volume_change_mohtat(sto,c_s_max):
    stoichpoints = np.array([0,0.12,0.18,0.24,0.50,1])
    thicknesspoints = np.array([0,2.406/100,3.3568/100,4.3668/100,5.583/100,13.0635/100])
    x = [sto]
    t_change = pybamm.Interpolant(stoichpoints, thicknesspoints, x, name=None, interpolator='linear', extrapolate=True, entries_string=None)
    return t_change

par_val = {}
# Room temp
par_val[0] = [4.0312e-08,1.8157e-07,1.0776,2.3586e-09,-4.9170e-09,-1.4406e-09,4.60788219e-16,4.56607447e-19]
Temp = 25
sno = 0
parameter_values.update(
    {
        # mechanical properties
        "Positive electrode Poisson's ratio": 0.3,
        "Positive electrode Young's modulus [Pa]": 375e9,
        "Positive electrode reference concentration for free of deformation [mol.m-3]": 0,
        "Positive electrode partial molar volume [m3.mol-1]": 7.28e-7,
        "Positive electrode volume change": nmc_volume_change_mohtat,
        # Loss of active materials (LAM) model
        "Positive electrode LAM constant exponential term": 2,
        "Positive electrode critical stress [Pa]": 375e6,
        # mechanical properties
        "Negative electrode Poisson's ratio": 0.2,
        "Negative electrode Young's modulus [Pa]": 15e9,
        "Negative electrode reference concentration for free of deformation [mol.m-3]": 0,
        "Negative electrode partial molar volume [m3.mol-1]": 3.1e-6,   
        "Negative electrode volume change": graphite_volume_change_mohtat,
        # Loss of active materials (LAM) model
        "Negative electrode LAM constant exponential term": 2,
        "Negative electrode critical stress [Pa]": 60e6,
        # Other
        "Cell thermal expansion coefficient [m.K-1]": 1.48E-6,
        "Lower voltage cut-off [V]": 3.0,
        # "Negative electrode active material volume fraction": eps_n_data,
        # "Positive electrode active material volume fraction": eps_p_data,
        "Initial temperature [K]": 273.15+Temp,
        "Ambient temperature [K]": 273.15+Temp,
        "SEI kinetic rate constant [m.s-1]":  par_val[sno][6], #1.08494281e-16 , 
        "EC diffusivity [m2.s-1]": par_val[sno][7],#8.30909086e-19,
        "SEI growth activation energy [J.mol-1]": 1.87422275e+04,#1.58777981e+04,
        "Initial inner SEI thickness [m]": 0e-09,
        "Initial outer SEI thickness [m]": 5e-09,
        "SEI resistivity [Ohm.m]": 30000.0,
        "Negative electrode partial molar volume [m3.mol-1]": 7e-06,
        # Initializing Particle Concentration
        # "Initial concentration in negative electrode [mol.m-3]": x100*parameter_values["Maximum concentration in negative electrode [mol.m-3]"],
        # "Initial concentration in positive electrode [mol.m-3]": y100*parameter_values["Maximum concentration in positive electrode [mol.m-3]"]
    },
    check_already_exists=False,
)

In [ ]:
# experiment
experiment = pybamm.Experiment(
    [
        "Charge at 1C until 4.2V",
        "Hold at 4.2 V until C/100",
        "Discharge at 1C until 3V",
    ]
)

In [ ]:
sim = pybamm.Simulation(
    model,
    experiment=experiment,
    parameter_values=parameter_values,
)
solution = sim.solve(initial_soc=0)

In [ ]:
# # plot
# plot = pybamm.QuickPlot(
#     solution,
#     [
#         "Negative particle concentration [mol.m-3]",
#         "Electrolyte concentration [mol.m-3]",
#         "Positive particle concentration [mol.m-3]",
#         "Current [A]",
#         "Negative electrode potential [V]",
#         "Electrolyte potential [V]",
#         "Positive electrode potential [V]",
#         "Voltage [V]",
#     ],
#     time_unit="seconds",
#     spatial_unit="um",
# )
# plot.dynamic_plot()

In [ ]:
vars = ["Time [s]", "Terminal voltage [V]", "Current [A]", "X-averaged negative particle surface concentration [mol.m-3]", "X-averaged positive particle surface concentration [mol.m-3]", "X-averaged negative particle concentration [mol.m-3]", "X-averaged positive particle concentration [mol.m-3]", "R-averaged negative particle concentration [mol.m-3]", "R-averaged positive particle concentration [mol.m-3]", "Average negative particle concentration [mol.m-3]", "Average positive particle concentration [mol.m-3]"]

In [ ]:
d1 = {}
for i,x in enumerate(vars):
    d1[i] = solution[x].entries
with open(data_DIR +'spm_new_fresh_vars.pickle', 'wb') as handle:
    pickle.dump(d1, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
# model
model = pybamm.lithium_ion.DFN(
    # {
    #     "SEI": "ec reaction limited",
    # }
)
parameter_values = pybamm.ParameterValues("Mohtat2020")

def nmc_volume_change_mohtat(sto,c_s_max):
    t_change = -1.10/100*(1-sto)
    return t_change

def graphite_volume_change_mohtat(sto,c_s_max):
    stoichpoints = np.array([0,0.12,0.18,0.24,0.50,1])
    thicknesspoints = np.array([0,2.406/100,3.3568/100,4.3668/100,5.583/100,13.0635/100])
    x = [sto]
    t_change = pybamm.Interpolant(stoichpoints, thicknesspoints, x, name=None, interpolator='linear', extrapolate=True, entries_string=None)
    return t_change

par_val = {}
# Room temp
par_val[0] = [4.0312e-08,1.8157e-07,1.0776,2.3586e-09,-4.9170e-09,-1.4406e-09,4.60788219e-16,4.56607447e-19]
Temp = 25
sno = 0
parameter_values.update(
    {
        # mechanical properties
        "Positive electrode Poisson's ratio": 0.3,
        "Positive electrode Young's modulus [Pa]": 375e9,
        "Positive electrode reference concentration for free of deformation [mol.m-3]": 0,
        "Positive electrode partial molar volume [m3.mol-1]": 7.28e-7,
        "Positive electrode volume change": nmc_volume_change_mohtat,
        # Loss of active materials (LAM) model
        "Positive electrode LAM constant exponential term": 2,
        "Positive electrode critical stress [Pa]": 375e6,
        # mechanical properties
        "Negative electrode Poisson's ratio": 0.2,
        "Negative electrode Young's modulus [Pa]": 15e9,
        "Negative electrode reference concentration for free of deformation [mol.m-3]": 0,
        "Negative electrode partial molar volume [m3.mol-1]": 3.1e-6,   
        "Negative electrode volume change": graphite_volume_change_mohtat,
        # Loss of active materials (LAM) model
        "Negative electrode LAM constant exponential term": 2,
        "Negative electrode critical stress [Pa]": 60e6,
        # Other
        "Cell thermal expansion coefficient [m.K-1]": 1.48E-6,
        "Lower voltage cut-off [V]": 3.0,
        # "Negative electrode active material volume fraction": eps_n_data,
        # "Positive electrode active material volume fraction": eps_p_data,
        "Initial temperature [K]": 273.15+Temp,
        "Ambient temperature [K]": 273.15+Temp,
        "SEI kinetic rate constant [m.s-1]":  par_val[sno][6], #1.08494281e-16 , 
        "EC diffusivity [m2.s-1]": par_val[sno][7],#8.30909086e-19,
        "SEI growth activation energy [J.mol-1]": 1.87422275e+04,#1.58777981e+04,
        "Initial inner SEI thickness [m]": 0e-09,
        "Initial outer SEI thickness [m]": 5e-09,
        "SEI resistivity [Ohm.m]": 30000.0,
        "Negative electrode partial molar volume [m3.mol-1]": 7e-06,
        # Initializing Particle Concentration
        # "Initial concentration in negative electrode [mol.m-3]": x100*parameter_values["Maximum concentration in negative electrode [mol.m-3]"],
        # "Initial concentration in positive electrode [mol.m-3]": y100*parameter_values["Maximum concentration in positive electrode [mol.m-3]"]
    },
    check_already_exists=False,
)

In [ ]:
sim = pybamm.Simulation(
    model,
    experiment=experiment,
    parameter_values=parameter_values,
)
solution = sim.solve(initial_soc=0)

In [ ]:
d1 = {}
for i,x in enumerate(vars):
    d1[i] = solution[x].entries
with open(data_DIR +'dfn_new_fresh_vars.pickle', 'wb') as handle:
    pickle.dump(d1, handle, protocol=pickle.HIGHEST_PROTOCOL)